In [ ]:
import os
import pandas as pd
import numpy as np

import requests
import zipfile
import io

# Reading LMP Data

Head to [Caiso OASIS](http://oasis.caiso.com/mrioasis/logon.do).

There are two documentations:
- [API Document 1](https://www.caiso.com/Documents/OASISAPISpecification.pdf) (2010)
- [API Document 2](http://www.caiso.com/Documents/OASIS-InterfaceSpecification_v5_1_1Clean_Fall2017Release.pdf) (2017)
- [Pnodes mapping](https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&ved=2ahUKEwi8_svGzsKEAxW15MkDHS6kBnkQFnoECC8QAQ&url=https%3A%2F%2Fwww.caiso.com%2FDocuments%2FFullNetworkModel_PricingNodeMapping_Reference.xls&usg=AOvVaw14Mjc4dr6DD_cKL19KvcfZ&opi=89978449)

## Downloading the data
We will follow [OASIA](http://www.caiso.com/Documents/OASIS-InterfaceSpecification_v5_1_1Clean_Fall2017Release.pdf) for the url query.

In [ ]:
oasiswebsite = "oasis.caiso.com"
context_path = "oasisapi"
url = f"http://{oasiswebsite}/{context_path}/SingleZip"

query_name = "PRC_RTPD_LMP"
url += f"?queryname={query_name}"

# we can only access 30 days at a time
startdate, enddate = "601", "701"
startdatetime=f"20230{startdate}T08:00-0000"
enddatetime=f"20230{enddate}T08:00-0000"
url += f"&startdatetime={startdatetime}&enddatetime={enddatetime}"

api_version = 2
url += f"&version={api_version}"

# This gives the data in csv. Default of `1` yields XML files
url += "&resultformat=6"

# market type
market_type = "RTPD" # {RTPD, DAM}
url += f"&market_run_id={market_type}"

# node we want to get query data for
node_id = "MIL1_3_PASGNODE"
url += f"&node={node_id}"

print(f"URL: {url}")

Let's download actual data and look into it

In [ ]:
def download_url(url):
    if input(f"Do you want to download:\n{url}?\n>> ").lower() in ["y", "yes"]:
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall(".")

In [ ]:
download_url(url)

In [ ]:
fname = os.path.join("20230601_20230701_PRC_RTPD_LMP_RTPD_20240224_11_52_30_v2.csv")
df = pd.read_csv(fname)
df

## Data Analysis
Now having our data, we shift to data analysis.

#### Static (ATLAS) data
So earlier we accessed `MIL1_3_PASGNODE`'s data. Which TAC does it belong to? Let's try to answer that below by downloading a map from pnodes to TAC.

In [ ]:
oasiswebsite = "oasis.caiso.com"
context_path = "oasisapi"
url = f"http://{oasiswebsite}/{context_path}/SingleZip"

query_name = "ATL_HUB" # get all trading hubs
query_name = "ATL_PNODE_MAP" # get all pnodes
query_name = "ATL_TAC_AREA_MAP" # get all TAC and pnodes
query_name = "ATL_APNODE" # get all aggregated pnodes
url += f"?queryname={query_name}"

# we can only access 30 days at a time
startdate, enddate = "601", "701"
startdatetime=f"20230{startdate}T08:00-0000"
enddatetime=f"20230{enddate}T08:00-0000"
url += f"&startdatetime={startdatetime}&enddatetime={enddatetime}"

# you have to do version 1 instead of 2... not sure why
api_version = 1
url += f"&version={api_version}"

url += "&resultformat=6"

# we do not need node or market_type since we are just asking for TAC data

In [ ]:
download_url(url)

In [ ]:
fname = os.path.join("20230601_20230701_ATL_TAC_AREA_MAP_N_20240224_12_01_37_v1.xml")
df = pd.read_xml(fname)
df

XML files are a little tricky. Let us use beautiful soup.

In [ ]:
from bs4 import BeautifulSoup

with open(fname, "r") as f:
    data = f.read()

bs_data = BeautifulSoup(data, "xml")

Just playing around with data and functions to see how to extract data.

In [ ]:
bs_data

atls_data_raw = bs_data.find_all('ATLS_DATA')

print(f"Number of elements: {len(atls_data_raw)}\n")

print(f"First entry:\n{atls_data_raw[0]}\n")

print(f"Getting TAC_AREA_NAME data: {atls_data_raw[0].find('TAC_AREA_NAME').text}")

# print(f"Getting TAC_AREA_NAME data: {atls_data_raw.find('TAC_AREA_NAME').text}")

Remove this since we found a way for the data to be returned as CSV (rather than XML)

In [ ]:
# def extract_xml_data(fname, tag, text_ids):
#     """ Opens XML file, finds all entries and extracts data into lists 
#     :param fname: xml file to open
#     :param tag: tag to find repeated entries of
#     :param text_ids: ids within each tag to extract data from
#     :return arr: 2d list with entries
#     """
#     with open(fname, "r") as f:
#         data = f.read()
# 
#     bs_data = BeautifulSoup(data, "xml")
#     data_raw = bs_data.find_all(tag)
#     assert len(data_raw) > 0, f"Did not find any data with tag {tag}"
#     
#     arr = np.empty(shape=(len(data_raw), len(text_ids)), dtype=object)
#     for i, data in enumerate(data_raw):
#         for j, text_id in enumerate(text_ids):
#             arr[i,j] = data.find(text_id).text
# 
#     return arr

In [ ]:
# arr = extract_xml_data(fname, "ATLS_DATA", ["TAC_AREA_NAME", "PNODE_NAME"])

Let's find the TAC for our pnode.

In [ ]:
# found_tac = None
# for (tac, pnode) in arr:
#     if pnode == node_id:
#         if found_tac is not None:
#             print(f"ruh roh, found multiple TACs to single pnode... (prev tac: {found_tac}")
#         found_tac = tac
# 
# print(f"pnode {node_id} belongs to {found_tac}")

This one is a little tricker to use, so we stop here.

# Pnodes, TAC, and REN

- Regions: CAISO has three regions: NP15, ZP26, and SP15. You can find more by googling "zp26 caiso" or by viewing the [full network model](http://www.caiso.com/Pages/DocumentsByGroup.aspx?GroupID=2d44e3d3-b489-449a-a805-915fa7608e29) (FNM)
- pnode: Pricing nodes. LMP data is taken at a specific `PNODE`.
- TAC: Transmission access charge. These are fees to pay for state's transmission system. Loads and demand are typically aggregated by TAC area.
- Balancing authorities. [More info](https://haas.berkeley.edu/wp-content/uploads/WP321Appendix.pdf)

### Transmission Access Charge (TAC)

This dataset gets us all the TAC areas. Think about this as aggregated areas of pnodes.

In [ ]:
fname = os.path.join("gym_examples", "envs", "dataset1", "20230917_20230918_ATL_TAC_AREA_MAP_N_20230921_10_30_28_v1.csv")
df = pd.read_csv(fname)
df

Let's try to find our pnode of interest.

In [ ]:
# node_id = "MIL1_3_PASGNODE"

# SF (TAC Area: TAC_NORTH)
node_id = "PAULSWT_1_N004" # load
node_id = "PAULSWT_1_N013" # gen
# node_id = "PAULSWT_1_GN002"  # gen

node_id = "FREMNT_1_N013"  # load

node_id = "EMBRCDR_2_N008" # load
node_id = "EMBRCDR_2_N010" # gen

node_id = "MOSSLND7_7_ND001" # load
node_id = "MOSSLND7_7_B2"    # gen

node_id = "DAIRYLND_7_N003" # load
node_id = "HOLLISTR_7_N001" # gen

# LA
# TAC Area: TAC_NORTH
node_id = "PERMNNTE_6_N002" # load

node_id = "COTWDPGE_1_N001" # load
node_id = "COTWDPGE_1_N024" # gen

# TAC Area: TAC_ECNTR
node_id = "ALAMT3G_7_N002" # load
node_id = "ALAMT3G_7_B1"   # gen

node_id = "ELLSGTY_1_N001" # load
node_id = "ELLIS_6_N003"   # gen

node_id = "OLY_SUB_LNODEOND" # load
node_id = "LCIENEGA_6_N005"  # gen

print(df[df["PNODE_ID"] == node_id])

### Zones

This dataset gets us all pnodes to AS (area). Think of AS as the largest subdivisions of California.

In [ ]:
fname = os.path.join("gym_examples", "envs", "dataset1", "20230917_20230918_ATL_PNODE_MAP_N_20230921_11_06_29_v1.csv")
# fname = "20230601_20230701_ATL_PNODE_MAP_N_20240224_13_13_05_v1.csv"
df = pd.read_csv(fname)
df

Let's find our pnode.

In [ ]:
print(df[df["PNODE_ID"] == node_id])

### Pnodes with complete information?

Did the pnode not have a listed TAC and/nor area? Let's find the pnodes one or the other, and then both.

In [ ]:
fname = os.path.join("gym_examples", "envs", "dataset1", "20230917_20230918_ATL_TAC_AREA_MAP_N_20230921_10_30_28_v1.csv")
df_tac = pd.read_csv(fname)
pnode_list_from_tac = df_tac["PNODE_ID"].unique()

fname = os.path.join("gym_examples", "envs", "dataset1", "20230917_20230918_ATL_PNODE_MAP_N_20230921_11_06_29_v1.csv")
df_area = pd.read_csv(fname)
pnode_list_from_area = df_area["PNODE_ID"].unique()

pnode_list_with_both_tac_and_area = list(set(pnode_list_from_tac).intersection(pnode_list_from_area))

Let's look at pnodes with a TAC value.

In [ ]:
for i, pnode in enumerate(pnode_list_from_tac):
    tac = df_tac[df_tac["PNODE_ID"] == pnode]["TAC_AREA_ID"].iloc[0]
    if tac != "TAC_LADWP":
        continue
    print("[%i] pnode=%s | tac=%s" % (i, pnode, tac))

Now let's consider those pnodes with area.

In [ ]:
for i, pnode in enumerate(pnode_list_from_area):
    area = df_area[df_area["PNODE_ID"] == pnode]["APNODE_ID"].iloc[0]
    print("[%i] pnode=%s | area=%s" % (i, pnode, area))

Let's look at pnodes with both.

In [ ]:
for i, pnode in enumerate(pnode_list_with_both_tac_and_area):
    tac = df_tac[df_tac["PNODE_ID"] == pnode]["TAC_AREA_ID"].iloc[0]
    area = df_area[df_area["PNODE_ID"] == pnode]["APNODE_ID"].iloc[0]
    if "ALAMT" not in pnode:
        continue
    print("[%i] pnode=%s | tac=%s | area=%s" % (i, pnode, tac, area))

Let's look just at pnodes.

# For your viewing pleasure

Here are a couple nice maps:
- Pretty LMP map: http://electricitymapper.appspot.com (src: https://github.com/emunsing/ElectricityMapper/tree/master)
- CAISO LMP map: http://www.caiso.com/pricemap/Pages/default.aspx
- Zone visualization: https://www.caiso.com/Documents/Chapter5_Inter-ZonalCongestionManagementMarket.pdf

### Trading Hubs (Zones)

Let us see our trading hubs.

In [ ]:
fname = "20230601_20230701_ATL_HUB_N_20240224_13_16_30_v1.csv"
df = pd.read_csv(fname)
df

This data set gets us all the pnodes.

In [ ]:
fname = os.path.join("gym_examples", "envs", "dataset1", "20230917_20230918_ATL_PNODE_N_20230921_12_16_19_v1.csv")
df = pd.read_csv(fname)
df[df["PNODE_ID"] == node_id]

Let  us get all the apnodes (aggregated)

In [ ]:
fname = "20230601_20230701_ATL_APNODE_N_20240224_13_20_06_v1.csv"
df = pd.read_csv(fname)
df